In [17]:
"""
CS4771 - Python for Machine Learning
Kaggle Competition Assignment 3=4
Todd Carter
V01187982
12-14-2025

"""


import torch
import torch.nn as nn
import math
from torch.utils.data import Dataset
from torch.utils.data import DataLoader
import pandas as pd
from sklearn.preprocessing import StandardScaler

# Class based on CustomGRU from LSTM_and_GRU.ipynb
class CustomGRU(nn.Module):
    def __init__(self, input_sz, hidden_sz):
        super().__init__()

        self.gru1 = GRULayer(input_sz, hidden_sz)
        self.gru2 = GRULayer(hidden_sz, hidden_sz)

        self.fc = nn.Sequential(
            nn.Linear(hidden_sz, 32),
            nn.ReLU(),
            nn.Linear(32, 1)
        )

    def forward(self, x):
        x = self.gru1(x)
        x = self.gru2(x)

        out = x[:, -1, :]
        return self.fc(out).squeeze(-1)



# Class to define the GRU layering to stop infinite loop
class GRULayer(nn.Module):
    def __init__(self, input_sz, hidden_sz):
        super().__init__()
        self.hidden_sz = hidden_sz
        self.weight_ih = nn.Linear(input_sz, 3 * hidden_sz)
        self.weight_hh = nn.Linear(hidden_sz, 3 * hidden_sz)

    def forward(self, x):
        # x: (batch, seq_len, input_sz)
        batch_sz, seq_len, _ = x.size()
        h_t = torch.zeros(batch_sz, self.hidden_sz, device=x.device)

        hidden_seq = []

        for t in range(seq_len):
            x_t = x[:, t, :]

            # 1. Compute all Input projections (x_t * W_ih)
            # Result shape: (batch, 3 * hidden_sz)
            x_gate = self.weight_ih(x_t)

            # 2. Compute all Hidden projections (h_{t-1} * W_hh)
            # Result shape: (batch, 3 * hidden_sz)
            h_gate = self.weight_hh(h_t)

            # Split them into specific gates: Reset(r), Update(z), New/Candidate(n)
            # PyTorch order is: Reset, Update, New
            x_r, x_z, x_n = x_gate.chunk(3, dim=1)
            h_r, h_z, h_n = h_gate.chunk(3, dim=1)

            # --- Gate Calculations ---

            # Reset Gate
            r_t = torch.sigmoid(x_r + h_r)

            # Update Gate
            z_t = torch.sigmoid(x_z + h_z)

            # Candidate Hidden State (n_t)
            # Note: PyTorch applies the Reset gate (r_t) to the HIDDEN projection (h_n)
            n_t = torch.tanh(x_n + (r_t * h_n))

            # --- Final State Update ---
            # PyTorch Logic: h_t = (1 - z) * n + z * h_old
            # (This implies z=1 means "Keep Old", z=0 means "Update New")
            h_t = (1 - z_t) * n_t + (z_t * h_t)
            
            hidden_seq.append(h_t.unsqueeze(1))

        return torch.cat(hidden_seq, dim=1)


# Class to produce a torch.tensor from the dataset
class ETTDataset(Dataset):
    def __init__(self, df, input_window=168):
        self.df = df
        self.input_window = input_window
        
        data = df[['HUFL','HULL','MUFL','MULL','LUFL','LULL','OT']].values
        self.X = data
        self.Y = data[:, -1]  # OT column

    def __len__(self):
        return len(self.X) - self.input_window

    def __getitem__(self, idx):
        x = self.X[idx : idx + self.input_window]      # shape: (window, 7)
        y = self.Y[idx + self.input_window]            # OT at next timestep
        return torch.tensor(x, dtype=torch.float32), torch.tensor(y, dtype=torch.float32)


#----------Data processing-------------

train_df = pd.read_csv("/kaggle/input/train-dataset-for-competition-4/train.csv")
test_df = pd.read_csv("/kaggle/input/testing-data-for-competition-4/test.csv")

features = ['HUFL','HULL','MUFL','MULL','LUFL','LULL']

scaler = StandardScaler()

train_df[features] = scaler.fit_transform(train_df[features])
test_df[features] = scaler.transform(test_df[features])

train_df['OT'] = scaler.fit_transform(train_df[['OT']])
#test_df['OT'] = scaler.transform(test_df[['OT']])

# Combine so sliding windows continue
full = pd.concat([train_df, test_df], ignore_index=True)
full['OT'] = full['OT'].ffill()  # use last known OT

# Hyperparameters
INPUT_SIZE = 7
HIDDEN_SIZE = 64
BATCH_SIZE = 10
SEQ_LEN = 168


#---------Training--------------

dataset = ETTDataset(train_df, input_window=168)
loader = DataLoader(dataset, batch_size=64, shuffle=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = CustomGRU(INPUT_SIZE, HIDDEN_SIZE).to(device)
criterion = nn.L1Loss()  # MAE
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

epochs = 10

for epoch in range(epochs):
    model.train()
    total_loss = 0
    for X, y in loader:
        X, y = X.to(device), y.to(device)

        optimizer.zero_grad()
        pred = model(X)
        loss = criterion(pred, y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    print(f"Epoch {epoch+1}/{epochs} - Loss: {total_loss/len(loader):.4f}")


print(f"--- GRU/LSTM Configuration ---")
print(f"Input: {INPUT_SIZE}, Hidden: {HIDDEN_SIZE}, Batch: {BATCH_SIZE}")


#-------------Prediction----------------

model.eval()
preds = []

with torch.no_grad():
    for i in range(len(train_df), len(full)):
        start = i - 168
        end = i

        window = full.loc[start:end-1, ['HUFL','HULL','MUFL','MULL','LUFL','LULL','OT']].values
        window = torch.tensor(window, dtype=torch.float32).unsqueeze(0).to(device)

        pred = model(window).item()
        preds.append(pred)

        # feed prediction back into OT column
        full.loc[i, 'OT'] = pred


# Submission file
sub = pd.read_csv("/kaggle/input/sample-data-for-competition-4/sample_submission.csv")
sub['OT'] = preds
sub.to_csv("submission.csv", index=False)


Epoch 1/10 - Loss: 0.1645
Epoch 2/10 - Loss: 0.0818
Epoch 3/10 - Loss: 0.0780
Epoch 4/10 - Loss: 0.0745
Epoch 5/10 - Loss: 0.0746
Epoch 6/10 - Loss: 0.0741
Epoch 7/10 - Loss: 0.0733
Epoch 8/10 - Loss: 0.0725
Epoch 9/10 - Loss: 0.0732
Epoch 10/10 - Loss: 0.0721
--- GRU/LSTM Configuration ---
Input: 7, Hidden: 64, Batch: 10
